In [3]:
# 一个完整的训练流程

import copy
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# =========================
# 1. 准备数据集
# =========================
torch.manual_seed(23)

class ToyDataset(Dataset):
    def __init__(self, count=100):
        self.features = torch.randn(count, 4, dtype=torch.float32)
        self.labels = (self.features[:, 0] > 0).long()

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


dataset = ToyDataset(100)

# 划分训练集、验证集、测试集：8:1:1
train_set, valid_set, test_set = random_split(
    dataset,
    [80, 10, 10],
    generator=torch.Generator().manual_seed(23)
)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=16, shuffle=False)
test_loader = DataLoader(test_set, batch_size=16, shuffle=False)

# =========================
# 2. 设置设备
# =========================
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("使用设备：", device)

# =========================
# 3. 构建模型和训练配置
# =========================
model = nn.Linear(4, 1).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 50
best_valid_loss = float("inf")
best_state = None

# =========================
# 4. 训练 + 验证
# =========================
for epoch in range(epochs):

    # ---- 训练阶段 ----
    model.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_samples = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.float().to(device).unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = x.size(0)
        train_loss_sum += loss.item() * batch_size
        train_correct += ((logits >= 0) == (y >= 0.5)).sum().item()
        train_samples += batch_size

    train_loss = train_loss_sum / train_samples
    train_acc = train_correct / train_samples

    # ---- 验证阶段 ----
    model.eval()

    valid_loss_sum = 0.0
    valid_correct = 0
    valid_samples = 0

    with torch.no_grad():
        for x, y in valid_loader:
            x = x.to(device)
            y = y.float().to(device).unsqueeze(1)

            logits = model(x)
            loss = criterion(logits, y)

            batch_size = x.size(0)
            valid_loss_sum += loss.item() * batch_size
            valid_correct += ((logits >= 0) == (y >= 0.5)).sum().item()
            valid_samples += batch_size

    valid_loss = valid_loss_sum / valid_samples
    valid_acc = valid_correct / valid_samples

    # 保存验证集 Loss 最低时的模型参数
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:02d} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.2%} | "
            f"valid_loss={valid_loss:.4f} | "
            f"valid_acc={valid_acc:.2%}"
        )

# =========================
# 5. 恢复最佳模型并保存
# =========================
model.load_state_dict(best_state)

torch.save(model.state_dict(), "toy_model.pth")
print("最佳模型已保存：toy_model.pth")

# =========================
# 6. 测试阶段
# =========================
model.eval()

test_loss_sum = 0.0
test_correct = 0
test_samples = 0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y.float().unsqueeze(1))

        preds = (logits >= 0).squeeze(1).long()

        test_loss_sum += loss.item() * x.size(0)
        test_correct += (preds == y).sum().item()
        test_samples += x.size(0)

print(f"Test Loss: {test_loss_sum / test_samples:.4f}")
print(f"Test Accuracy: {test_correct / test_samples:.2%}")

# =========================
# 7. 加载模型并进行推理
# =========================
loaded_model = nn.Linear(4, 1).to(device)

state_dict = torch.load(
    "toy_model.pth",
    map_location=device,
    weights_only=True
)
loaded_model.load_state_dict(state_dict)
loaded_model.eval()

# 模拟两条新样本，每条有 4 个特征
new_x = torch.tensor([
    [2.0, 0.1, -0.5, 1.0],
    [-1.0, 0.2, 0.3, -0.4]
], dtype=torch.float32).to(device)

with torch.no_grad():
    logits = loaded_model(new_x)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).long()

print("预测概率：", probabilities.squeeze(1).cpu())
print("预测类别：", predictions.squeeze(1).cpu())


使用设备： mps
Epoch 10 | train_loss=0.5794 | train_acc=76.25% | valid_loss=0.5560 | valid_acc=80.00%
Epoch 20 | train_loss=0.4600 | train_acc=96.25% | valid_loss=0.4612 | valid_acc=90.00%
Epoch 30 | train_loss=0.3840 | train_acc=97.50% | valid_loss=0.3969 | valid_acc=90.00%
Epoch 40 | train_loss=0.3320 | train_acc=97.50% | valid_loss=0.3487 | valid_acc=90.00%
Epoch 50 | train_loss=0.2943 | train_acc=97.50% | valid_loss=0.3153 | valid_acc=90.00%
最佳模型已保存：toy_model.pth
Test Loss: 0.1884
Test Accuracy: 100.00%
预测概率： tensor([0.9658, 0.1442])
预测类别： tensor([1, 0])
